In [2]:

using Measurements
import Measurements: uncertainty, value

using CairoMakie
using Observables
#Mittelwert bilden
using CSVFiles, DataFrames
using CSV, Statistics
#gewichteter Mittelwert isst array measurement und spuckt wert aus 
function gew_Mtw(m)
    val= value.(m)
    unct = uncertainty.(m)
    x=0
    for i in 1:length(val)
        x=x+val[i]*(unct[i])^(-2)
    end
    v=0
    for i in 1:length(val)
        v=v+(unct[i])^(-2)
    end
    gewmtw=x/v
    return gewmtw
end

#gewichtete standartabweichung extern
function gew_Stw_ext(m, mtw) 
    val= value.(m)
    unct = uncertainty.(m)
    x=0
    for i in 1:length(val)
        x=x+(val[i]-mtw)^2*(unct[i])^(-2)
    end
    v=0
    for i in 1:length(val)
        v=v+(unct[i])^(-2)
    end
    gewstw=sqrt(x/(v*(length(val)-1)))
    return gewstw
end
#gewichtete Standartabweichung intern
function gew_Stw_int(m)
    unct = uncertainty.(m)
    v=0
    for i in 1:length(unct)
        v=v+(unct[i])^(-2)
    end
    v=1/sqrt(v)
end

function first_significant_digit(x::Real)
    x == 0 && return 0  # Sonderfall: Wenn x genau 0 ist, gibt es keine signifikante Stelle, also gib 0 zurück.

    absx = abs(x)  # Der Betrag der Zahl wird genommen, damit negative Vorzeichen ignoriert werden.

    # Berechne die Zehnerpotenz, in der sich die erste signifikante Stelle befindet.
    # log10(absx) gibt den dekadischen Logarithmus. floor(Int, ...) rundet nach unten auf ganze Zahl.
    exponent = floor(Int, log10(absx))

    # Teile absx durch 10^exponent, um die Zahl in den Bereich [1,10) zu bringen
    # Beispiel: aus 456 → 4.56, aus 0.00456 → 4.56
    significand = absx / 10.0^exponent

    # Jetzt holen wir die ganzzahlige Ziffer vor dem Komma, das ist die erste signifikante Ziffer
    return floor(Int, significand)
end


"""
    round_measurement(m::Measurement; sigdigits_error::Int=1)

Rundet erst die Unsicherheit von `m` auf `sigdigits_error` signifikante Stellen
und rundet dann den Messwert auf dieselbe Genauigkeit (Dezimalstellen),
liefert einen neuen `Measurement`. Sigdigits_error wird automatisch auf den richtigen wert gesetzt durch if bedingung 
wenn first sig digit =1 oder 2, wird eine weitere stelle hinzugenommen.

input: ein measurement, output: ein measurement. Kann punktweise angewendet werden
"""
function round_measurement(m::Measurement)
    # 1) Unsicherheit extrahieren und auf sigdigits_error sig. Stellen runden
    u = uncertainty(m)
    if first_significant_digit(u)<3
        sigdigits_error=2
    else
        sigdigits_error=1
    end
    #println(sigdigits_error)

    u_r = round(u; sigdigits=sigdigits_error)

    
    if u_r==0 #wenn gauß versagt, normalverteilung und stw
        

    # Ziehe N Stichproben aus einer Normalverteilung und berechne KINETISCHE ENERGIE!!!
    p_samples = randn(100_000) .* 0.05  # p ~ N(0, σp=0.05) kein plan warum, aber die 0.05 sind auf jkeden fall der fehler vorher

    # Berechne kinetische Energie für jede Probe
    E_samples = p_samples.^2 ./ (2)

    # Mittelwert und Standardabweichung der Energie
    E_mean = mean(E_samples)
    E_std = std(E_samples)
    
    u_r = round(E_std; sigdigits=sigdigits_error)
        #println(u_r)
    end
    
    
    
    
    
    # 2) Anzahl Dezimalstellen bestimmen:
    #    Wenn u_r = x * 10^e  (mit 1 ≤ x < 10), dann ist e = floor(log10(u_r))
    #    und wir benötigen -e  Dezimalstellen (für e ≤ 0)
    e = floor(Int, log10(u_r))
       # println(e)
    if first_significant_digit(u)<3
        dec = max(0, -e) +1
    else
        dec = max(0, -e)
    end
    #dec = max(0, -e)
    #println(dec, log(12,u_r))

    # 3) Wert runden und neuen Measurement erstellen
    v_r = round(value(m); digits=dec)
    return measurement(v_r, u_r)
end

# Beispiel
m = measurement(0.023456, 0.0120236789)
println("Original: ", value(m))                  # 1.23456 ± 0.06789
m2 = round_measurement(m)
println("Gerundet: ", value(m2))  # z.B. 1.23(7) → ±0.07, Wert 1.23

#messung = (DataFrame(load("Messung_schwebung.csv")))




function nice_ticks_and_labels(x; n_ticks=20, n_labels=21, sigdigit=8)
    order = Float64(floor(Int, log10(maximum(x))))
    n = Float64(floor(Int, maximum(x) / 10^order))+1  # erste Ziffer
    tick_max = n * 10^order

    # Tick-Positionen: 0, 1*10^order, 2*10^order, ..., n*10^order
    ticks = round.(collect((0:100/n_ticks*10^(order-2):tick_max)) ; digits =sigdigit)  # feinere Tick-Striche

    # Beschriftete Zahlen: 10 gleichmäßig
    labels = collect(range(0, stop=tick_max, length=n_labels))

    return ticks, labels
end

function alignedlabels(ticks, labels; atol=0.01)
    aligned = String[]
    for x in ticks
        match = findfirst(lbl -> isapprox(x, lbl; atol=atol, rtol=0), labels)
        if match === nothing
            push!(aligned, "")
        else
            push!(aligned, string(labels[match]))
        end
    end
    return aligned
end






x = value.(th)##ANPASSEN##
y = root ##ANPASSEN##
#Fehler-Array mit gleichem Fehler
Δx= delth
Δy= 0   
#=for i in 1:length(x)
    push!(Δx,ΔV) ###ANPASSEN##
    push!(Δy,ΔT) ##ANPASSEN##
end=#
#=for i in 1:length(x)
    push!(Δx,ΔV) ###ANPASSEN##
    push!(Δy,ΔT) ##ANPASSEN##
end=#

with_theme(theme_latexfonts()) do
    fig = Figure()
    ax = Axis(fig[1,1])
    #Fehlerbalken
    #errorbars!(ax, x, y, Δy, color=:grey, whiskerwidth = 5)
    errorbars!(ax, x, y, Δx, color=:grey, whiskerwidth = 5, direction = :x)
    #Messwerte
    scatter!(ax,x, y, markersize = 6, color=:black)
    
    ax.title = L"$Geradenanpassung$" ##ANPASSEN##


    #Skalierung der x-Achse ##ANPASSEN##
xticks = nice_ticks_and_labels(x, n_labels=21, n_ticks=40)[1]
xticklabels = nice_ticks_and_labels(x, n_labels=21, n_ticks=40)[2]
aligned_xlabels = alignedlabels(xticks, xticklabels)
#Skalierung der y-Achse ##ANPASSEN##
yticks = nice_ticks_and_labels(y, n_labels=21)[1]
yticklabels = nice_ticks_and_labels(y, n_labels=21)[2]   
aligned_ylabels = alignedlabels(yticks, yticklabels)
ax.xlabel = L" $sin(\theta)$" ##ANPASSEN##
ax.ylabel = L"$\sqrt(h^2 +k^2 +l^2)$  " ##ANPASSEN##

ax.xticks = (xticks, aligned_xlabels)
ax.yticks = (yticks, aligned_ylabels) 

#axislegend(ax; position=(:left, :top))
    #plot speichern
    #save("Messung_E_A.png", fig, px_per_unit = 2) ##ANPASSEN##

    fig
end
#rechnerische Geradenanpassung Eisen
T =  value.(x_eisen) #x-werte
L_E =  value.(R_eisen) #y-werte
T_=sum(T)
L_ = sum(L_E)
TT_ = sum(T.*T)
TL_ = sum(T.*L_E)
Δ = length(T)*TT_-T_*T_
a_E= 1/Δ*(length(T)*TL_-T_*L_)
b= 1/Δ*(TT_*L_-T_*TL_)
ΔL2=1/(length(T)-2)*sum((a_E.*T.+b.-L_E).^2)
Δa_E = sqrt(ΔL2*length(T)/Δ)
Δb = sqrt(ΔL2*TT_/Δ)
a_E, Δa_E
a_m_eisen = round_measurement(measurement(a_E, Δa_E))
b_m_eisen = round_measurement(measurement(b, Δb))


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
ERROR: LoadError: Error opening package file C:\Users\va\.julia\compiled\v1.12\FFMPEG_jll\uSD0T_L36uC.dll: Eine Anwendungssteuerungsrichtlinie hat diese Datei blockiert. 

Stacktrace:
  [1] _include_from_serialized(pkg::Base.PkgId, path::String, ocachepath::String, depmods::Vector{Any}; register::Bool)
    @ Base .\loading.jl:1270
  [2] _include_from_serialized
    @ .\loading.jl:1246 [inlined]
  [3] _require_search_from_serialized(pkg::Base.PkgId, sourcepath::String, build_id::UInt128, stalecheck::Bool; reasons::Dict{String, Int64}, DEPOT_PATH::Vector{String})
    @ Base .\loading.jl:2087
  [4] _require_search_from_serialized
    @ .\loading.jl:1981 [inlined]
  [5] __require_prelocked(pkg::Base.PkgId, env::String)
    @ Base .\loading.jl:2599
  [6] _require_prelocked(uuidkey::Base.PkgId, env::String)
    @ Base .\loading.jl:2465
  [7] macro expansion
    @ .\loading.jl:2393 [inlined]
 

ErrorException: Failed to precompile CairoMakie [13f3f980-e62b-5c42-98c6-ff1f3baf88f0] to "C:\\Users\\va\\.julia\\compiled\\v1.12\\CairoMakie\\jl_9748.tmp".